# PRAGMA Phase 2 - Fit the structured processor

Fits `PragmaProcessor` (`src/pragma/processing/processor.py`) on the train-split evaluation records from `001_point_in_time_records.ipynb`, and inspects the resulting vocabularies, bucket boundaries, and OOV/coverage statistics required by the Phase 2 exit gate (section 17 of the implementation plan): *token distributions, OOV, truncation, and per-key coverage are reviewed; no validation/test data was used to fit processor artifacts.*

See ADR 0003 (fitted processor vs. HF tokenizer) and ADR 0008 (token ID space: global numeric buckets, per-key categorical vocab, shared BPE, milestone presence tokens).

In [1]:
import json
from pathlib import Path

import pandas as pd

from pragma.config import ProcessorConfig
from pragma.data.records import PointInTimeRecordBuilder, SplitConfig, drop_zero_event_records
from pragma.processing import PragmaProcessor
from pragma.schema import SchemaRegistry

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSOR_DIR = REPO_ROOT / "data" / "processor"
PROCESSOR_DIR

WindowsPath('C:/Users/levyr/Desktop/random-projects/pragma/data/processor')

## Rebuild evaluation records (same deterministic split as notebook 001)

In [2]:
events_df = pd.read_parquet(RAW_DIR / "events.parquet")
profile_df = pd.read_parquet(RAW_DIR / "profile_state.parquet")

registry = SchemaRegistry.default()
records = PointInTimeRecordBuilder(registry, SplitConfig()).build(events_df, profile_df)
records, n_dropped = drop_zero_event_records(records)
print(f"Dropped {n_dropped} zero-event records before fitting (ADR 0014)")
n_train = sum(r.split == "train" for r in records)
print(f"{n_train} / {len(records)} records are in the train split")

Dropped 15 zero-event records before fitting (ADR 0014)
385 / 485 records are in the train split


## Fit the processor on the train split only

In [3]:
config = ProcessorConfig.load(REPO_ROOT / "configs" / "processor" / "default.json")
processor = PragmaProcessor(registry, config)
report = processor.fit(records)

print(f"n_train_records: {report.n_train_records}")
print(f"total_vocab_size: {report.total_vocab_size}")
print(f"text_local_vocab_size (shared BPE): {report.text_local_vocab_size}")
print(f"text_oov_rate: {report.text_oov_rate:.4f} (byte-level BPE, expected ~0 per ADR 0008)")

n_train_records: 385
total_vocab_size: 412
text_local_vocab_size (shared BPE): 338
text_oov_rate: 0.0000 (byte-level BPE, expected ~0 per ADR 0008)


## Vocabulary size breakdown

Special tokens + key tokens (one per tokenizable/milestone key) + numeric buckets (global, shared across numeric keys) + categorical values (namespaced per key) + shared BPE subwords.

In [4]:
special_count = processor.special_tokens.count
key_count = len(processor.key_vocab)
numeric_count = processor.numeric_bucketizer.vocab_size
categorical_count = sum(
    processor.categorical_encoder.vocab_size(k)
    for k in processor.categorical_encoder.fitted_keys()
)
text_count = report.text_local_vocab_size

breakdown = pd.Series(
    {
        "special_tokens": special_count,
        "key_tokens": key_count,
        "numeric_buckets (global)": numeric_count,
        "categorical_values (per-key)": categorical_count,
        "bpe_subwords (shared)": text_count,
    },
    name="n_tokens",
)
breakdown.to_frame().assign(share=lambda d: d["n_tokens"] / d["n_tokens"].sum())

,n_tokens,share
special_tokens,7,0.016990
key_tokens,20,0.048544
numeric_buckets (global),17,0.041262
categorical_values (per-key),30,0.072816
bpe_subwords (shared),338,0.820388


## Numeric bucket boundaries (train-fitted, per key)

In [5]:
for key in processor.numeric_bucketizer.fitted_keys():
    stats = processor.numeric_bucketizer.stats_for(key)
    print(f"{key}: n_train_values={stats.n_train_values}, "
          f"n_zero_values={stats.n_zero_values}, boundaries={stats.boundaries}")

amount: n_train_values=66571, n_zero_values=723, boundaries=[1.92, 3.06, 4.22, 5.44, 6.81, 8.38, 10.100624999999999, 12.27, 14.81, 17.92, 21.878125, 27.25, 35.29, 48.42, 75.886875]


## Categorical coverage and OOV rates

OOV rate here reflects only values *seen during fitting but never looked up again* (fit doesn't call transform) — it will be exercised properly once `003_tokenize_shards.ipynb` transforms held-out val/test records against these train-fitted vocabularies.

In [6]:
pd.Series(report.categorical_oov_rates, name="oov_rate").sort_values(ascending=False).to_frame()

,oov_rate
type,0.0
currency,0.0
direction,0.0
channel,0.0
mcc,0.0
view,0.0
source_currency,0.0
country,0.0
plan,0.0
kyc_level,0.0


In [7]:
sample_key = "currency"
vocab = processor.categorical_encoder.stats_for(sample_key).vocab
print(f"'{sample_key}' vocab ({len(vocab)} values): {vocab}")

'currency' vocab (6 values): {'EUR': 44, 'GBP': 45, 'ISK': 46, 'PLN': 47, 'RON': 48, 'USD': 49}


## Held-out OOV check

Transform every val/test record and measure OOV against the train-only vocabularies — this is the number that actually matters for the Phase 2 exit gate, since fitting never sees these records.

In [8]:
val_test_records = [r for r in records if r.split != "train"]
for r in val_test_records:
    processor.transform(r)

held_out_oov = {
    key: processor.categorical_encoder.stats_for(key).oov_rate
    for key in processor.categorical_encoder.fitted_keys()
}
pd.Series(held_out_oov, name="held_out_oov_rate").sort_values(ascending=False).to_frame()

,held_out_oov_rate
age_band,1.0
balance_quantile,1.0
country,1.0
kyc_level,1.0
is_active,1.0
plan,1.0
channel,0.0
currency,0.0
direction,0.0
mcc,0.0


## Transform one record and inspect the tokenized output

In [9]:
example = next(r for r in records if r.split == "train" and r.events_before_evaluation)
tokenized = processor.transform(example)

print(f"entity: {tokenized.entity_id} ({tokenized.split})")
print(f"n profile fields: {len(tokenized.profile_fields)}")
print(f"n events kept / total: {tokenized.n_events_kept} / {tokenized.n_events_total}")

first_event = tokenized.events[0]
print(f"\nfirst event key_ids: {first_event.key_ids()}")
print(f"first event value_ids: {first_event.value_ids()}")
print(f"first event within_field_pos: {first_event.within_field_pos()}")

entity: u000000 (train)
n profile fields: 10
n events kept / total: 219 / 219

first event key_ids: [8, 12, 14, 13, 13, 13, 10]
first event value_ids: [41, 45, 50, 354, 352, 355, 55]
first event within_field_pos: [0, 0, 0, 0, 1, 2, 0]


## Save the fitted processor bundle

Mirrors `scripts/fit_processor.py` — reused by notebook 003.

In [10]:
PROCESSOR_DIR.mkdir(parents=True, exist_ok=True)
processor.save(PROCESSOR_DIR)
(PROCESSOR_DIR / "fit_report.json").write_text(
    json.dumps(
        {
            "n_train_records": report.n_train_records,
            "numeric_stats": report.numeric_stats,
            "categorical_oov_rates": report.categorical_oov_rates,
            "text_oov_rate": report.text_oov_rate,
            "text_local_vocab_size": report.text_local_vocab_size,
            "total_vocab_size": report.total_vocab_size,
        },
        indent=2,
        default=str,
    )
)
print(f"Bundle written to {PROCESSOR_DIR}")

Bundle written to C:\Users\levyr\Desktop\random-projects\pragma\data\processor


## Round-trip check

Reload the bundle from disk and confirm every record tokenizes to byte-identical output — the Phase 2 exit gate's core guarantee: *a model checkpoint is invalid without the exact processor bundle used to create its token IDs.*

In [11]:
reloaded = PragmaProcessor.load(PROCESSOR_DIR, registry)
mismatches = [
    r.entity_id
    for r in records
    if processor.transform(r).to_dict() != reloaded.transform(r).to_dict()
]
print(f"mismatched records after save/load round trip: {len(mismatches)}")
assert not mismatches

mismatched records after save/load round trip: 0
